# Collect results
Uncomment the benchmarks and methods you want. The run cell evaluates **every selected method on every selected benchmark**.

In Colab, select a GPU runtime for GPU methods and run Setup; it clones the public repository automatically. Automatic cloning requires public GitHub access. Alternatively, upload a `decision-rerank-colab*.tar.gz` bundle in the Files panel before Setup. Locally, open this notebook from the checkout.

Setup keeps the installed NumPy/SciPy pair and CUDA PyTorch, so a fresh Colab session should not need a restart. If an earlier Setup already replaced loaded NumPy/SciPy, choose **Runtime → Restart session** once, then rerun Setup.


In [ ]:
#@title Setup
import os, runpy, subprocess, tarfile
from pathlib import Path

archive = next(Path.cwd().glob('decision-rerank-colab*.tar.gz'), None)
if archive:
    with tarfile.open(archive) as bundle:
        bundle.extractall(filter='data')
setup = next((p for p in [Path('tools/colab.py'), Path('../tools/colab.py'), Path('decision-rerank/tools/colab.py')] if p.exists()), None)
if setup is None:
    try:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ellimilial/decision-rerank.git', 'decision-rerank'],
                       check=True, capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f"{error.stderr.strip()}\n"
            "Colab needs public access to this GitHub repository. Make it public, or upload "
            "decision-rerank-colab.tar.gz in the Files panel and rerun Setup."
        ) from None
    setup = Path('decision-rerank/tools/colab.py')
runpy.run_path(str(setup))['setup_notebook']()


In [ ]:
from decision_rerank.collection import benchmark, method, collect_results, download_results


In [ ]:
# Optional Hugging Face token. Skip this cell or press Enter to keep the current setting.
import os
from getpass import getpass

token = getpass("HF_TOKEN (Enter to skip): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
del token


## Benchmarks
Comment out any selections you do not want.


In [ ]:
BENCHMARKS = [
    benchmark('nfcorpus_dev', panel='nfcorpus_pilot', limit=5),  # Same 5-query pilot
    # benchmark('nfcorpus_dev', panel='nfcorpus_pilot'),        # 20-query pilot
    # benchmark('bioasq13b_curated', panel='bioasq'),             # 30-query panel
    # benchmark('curev1_en', panel='cure'),                      # 30-query panel
    # benchmark('bioasq13b_curated'),                            # Full 340 queries
    # benchmark('curev1_en'),                                   # Full 2,000 queries
    # benchmark('nfcorpus_dev'),                                # Full 324 dev queries
]


## Methods
Uncomment any combination. Methods run in list order, each across all selected benchmarks. GPU choices are roughly ordered from larger to smaller models, followed by CPU methods; actual memory and runtime depend on model format, input length and batch size.

Missing weights download automatically. Hosted methods use paid APIs and read keys from your environment, Colab Secrets or a hidden prompt. TF-IDF is a lexical control fitted to each supplied candidate list.

Kev-4B merges its LoRA on CPU before moving to GPU; it defaults to FP16 and batch size 1. Full-model GPU fit remains to be checked.


In [ ]:
METHODS = [
    # GPU: roughly heavier to lighter by model size
    # method('biodecision'),     # 4B: lighteternal/biodecision-v2-4b
    # method('qwen35-base'),     # 4B: BioDecision backbone control
    # method('kev-4b'),          # 4B: direct GPU loading, batch size 1
    # method('qwen3-4b'),        # Qwen/Qwen3-Reranker-4B
    # method('jina-v35'),        # 0.6B: joint list, bounded attention memory
    # method('medcpt'),

    # Lexical controls (CPU)
    # method('tfidf-cpu'),
    method('bm25'),

    # Hosted decisions: no local GPU; paid API calls
    # method('jev'),
    # method('openai-decisions'),
]


## Run
Selected hosted methods check their keys **before any model runs**:

- JEV and OpenAI Luna Decisions: `OPENROUTER_API_KEY`

Existing environment variables or Colab Secrets are reused. Otherwise, enter the key in the hidden prompt. Keys stay in runtime memory and are not written into the notebook or result files. CPU/GPU-only selections need no keys.

Results are saved after each query. Reuse the output folder to resume. For Luna, use a fresh output folder if earlier results used the direct OpenAI API.


In [ ]:
archive = collect_results(BENCHMARKS, METHODS, output='results')


In [ ]:
download_results(archive)
